# VT-GraF Benchmark: Visible--Thermal Granular Faults under severe clutter
## Comparison: Unsupervised Generalized Frangi Graph vs. Standard Frangi + SAM 2

This notebook reproduces the comparison on the **VT-GraF-Dataset** (multimodal asphalt crack dataset with severe granular clutter) between:
1. **Ours (Generalized Frangi Graph)**: Our unsupervised multimodal graph-based method (using MST + Betweenness Centrality on GPU).
2. **Baseline (Standard Frangi on Thermal + SAM 2 on Visible)**: Zero-shot SAM 2 prompted automatically by standard Frangi filter responses from the thermal modality.

---
### Technical Setup:
- Automatically downloads the **VT-GraF-Dataset** from Google Drive.
- Installs **Segment Anything 2 (SAM 2)** from Meta's repository and downloads the `sam2_hiera_large.pt` weights.
- Imports the custom GPU graph extraction package.
- Runs both methods and outputs comparative metrics: Jaccard (IoU), Tversky index, and Wasserstein distance.


## 1. Environment Setup & Datasets
We install the required libraries (including Meta's SAM 2 repository) and retrieve the pre-trained weights.


In [ ]:
# Install required packages
!pip install -q gdown POT scikit-image opencv-python matplotlib scipy torch torchvision

# Install SAM 2 from Meta's source
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Download SAM 2 Large weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt


### Download the VT-GraF-Dataset
If the dataset is not already present, we download it from Google Drive using `gdown`.


In [ ]:
import os
import gdown
from pathlib import Path

folder_id = '1d79CVf9Vqgwwjqn6b2gbc40eu2MM7B7-'
dest_dir = 'VT-GraF-Dataset'

def check_dataset_exists():
    for path in Path('.').rglob('Fissure 1'):
        return True
    return False

if not check_dataset_exists():
    print("Downloading VT-GraF-Dataset from Google Drive...")
    gdown.download_folder(id=folder_id, output=dest_dir, quiet=False, use_cookies=False)
    print("Download completed.")
else:
    print("Dataset already present.")


## 2. Load the Generalized Frangi Graph Modules
We append the package path (handling both local repository execution and Google Colab environments) and load the required modules.


In [ ]:
import sys
from pathlib import Path

# 1. Add ISPRS folder to sys.path
if not Path('ISPRS/src').exists():
    print("Running in Colab: Cloning repository to access code modules...")
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset.git
    !cd Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset && git sparse-checkout set ISPRS
    sys.path.insert(0, 'Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS')
else:
    print("Running locally in repository.")
    sys.path.insert(0, 'ISPRS')

# 2. Import package modules
from src import (
    VTGraFDataset, extract_frangi_graph_gpu, skeletonize_lee,
    thicken, compute_metrics, wasserstein_distance_skeletons
)
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 3. Initialize dataset loader
dataset = VTGraFDataset('.')


## 3. Initialize SAM 2 Predictor
We load the pre-trained SAM 2 Large model on GPU (or CPU as fallback) and initialize the image predictor.


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

predictor = SAM2ImagePredictor(build_sam2(model_cfg, sam2_checkpoint, device=device))
print("SAM 2 Large predictor loaded successfully!")


## 4. Define SAM 2 Baseline Prompter
We define the baseline approach which extracts points from standard thermal Frangi responses and queries SAM 2 on the high-resolution visible image.


In [ ]:
import numpy as np
import cv2
from skimage.filters import frangi

def run_baseline_frangi_sam2(sample, predictor, num_prompts=12):
    # Retrieve the images
    img_vis_t = sample['visible'] # Torch tensor [0, 1]
    img_ir_t  = sample['infrared'] # Torch tensor [0, 1]
    
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_ir_np = (img_ir_t.numpy() * 255).astype(np.uint8)
    
    # 1. Apply standard Frangi on the thermal image
    # Note: We use sigmas=[20, 30, 40] to match our method's scales exactly
    frangi_response = frangi(img_ir_np / 255.0, sigmas=[20, 30, 40])
    
    # 2. Threshold the response to get a binary mask
    # We take the top 0.5% intensity pixels to represent the crack
    thresh = np.percentile(frangi_response, 99.5)
    binary_mask = frangi_response > thresh
    
    # 3. Skeletonize to get a thin centerline
    skel = skeletonize_lee(binary_mask)
    
    # 4. Sample point coordinates along the skeleton
    y_coords, x_coords = np.where(skel > 0)
    total_pts = len(y_coords)
    
    if total_pts < 3:
        # Fallback if the Frangi response is empty: sample from max intensity pixel
        y_max, x_max = np.unravel_index(np.argmax(frangi_response), frangi_response.shape)
        pts = np.array([[x_max, y_max]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        # Regular spatial sampling
        step = max(1, total_pts // num_prompts)
        pts_x = x_coords[::step][:num_prompts]
        pts_y = y_coords[::step][:num_prompts]
        pts = np.column_stack((pts_x, pts_y)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32) # 1 means positive points
        
    # 5. Predict using SAM 2 on the visible image
    # Convert visible grayscale to RGB (3-channels) for SAM 2
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    predictor.set_image(img_vis_rgb)
    
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts,
        'frangi_response': frangi_response,
        'binary_mask': binary_mask,
        'skel': skel
    }


## 5. Quantitative Evaluation & Benchmark
We evaluate both methods on all 5 images of the VT-GraF-Dataset.
- For **Ours (Generalized Frangi Graph)**, we use parameters: `K=2` (dual triangle graph), scale set `Σ=[20,30,40]` and weights `visible: 1/3, infrared: 2/3`. Skeletons are thickened to **5 pixels** for final evaluation.
- For the **Baseline (Frangi Thermal + SAM 2)**, we generate the mask, skeletonize it, and thicken it to **5 pixels** similarly.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

results = []
visualizations = {}

# Set weights to match the latest notebook: 1/3 Visible, 2/3 Infrared
weights_ours = {'visible': 1/3, 'infrared': 2/3}

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OURS (Generalized Frangi Graph) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    
    # Run the latest code parameters: K=2 and multiscale Σ=[20,30,40]
    max_S_global, sim_img, centrality_i, timings_i, diagnostics_i = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=[20, 30, 40], R=3, K=2, device=device
    )
    
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    # Post-process: skeletonize and thicken to 5 pixels (matching latest notebook)
    sk_pred_ours = skeletonize_lee(pred_ours)
    sk_pred_thick_ours = thicken(sk_pred_ours, pixels=5)
    
    # --- 2. RUN BASELINE (Frangi Thermal + SAM 2) ---
    baseline_outputs = run_baseline_frangi_sam2(sample, predictor, num_prompts=12)
    pred_baseline = baseline_outputs['pred_mask']
    sampled_pts = baseline_outputs['pts']
    sk_pred_baseline = skeletonize_lee(pred_baseline)
    sk_pred_thick_baseline = thicken(sk_pred_baseline, pixels=5)
    
    # --- 3. GROUND TRUTH SKELETON ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 4. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_pred_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_pred_thick_ours, sk_gt_thick)
    
    # Baseline
    jac_base, tv_base = compute_metrics(sk_pred_thick_baseline, sk_gt_thick)
    wass_base = wasserstein_distance_skeletons(sk_pred_thick_baseline, sk_gt_thick)
    
    results.append({
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours,
        'Baseline_Jaccard': jac_base,
        'Baseline_Tversky': tv_base,
        'Baseline_Wasserstein': wass_base
    })
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours_pred': pred_ours,
        'ours': sk_pred_thick_ours,
        'ours_centrality': centrality_i,
        'ours_fused_frangi': max_S_global,
        'ours_similarity': sim_img,
        'ours_tau_mask': diagnostics_i.get('tau_mask', np.zeros_like(pred_ours)),
        'ours_comp_mask': diagnostics_i.get('comp_mask', np.zeros_like(pred_ours)),
        'baseline': sk_pred_thick_baseline,
        'baseline_mask': pred_baseline,
        'points': sampled_pts,
        'baseline_frangi': baseline_outputs['frangi_response'],
        'baseline_binary': baseline_outputs['binary_mask'],
        'baseline_skel': baseline_outputs['skel']
    }

df_res = pd.DataFrame(results)


## 6. Comparison Results Table
We display the quantitative results and compute the mean scores across all fissures.


In [ ]:
# Format and display the results
pd.set_option('display.precision', 4)
display(df_res)

# Print Global Averages
print("\n" + "="*50)
print("--- SUMMARY STATISTICS ---")
print("="*50)
print(f"Jaccard (IoU) Mean   : Ours = {df_res['Ours_Jaccard'].mean():.4f} | Baseline = {df_res['Baseline_Jaccard'].mean():.4f}")
print(f"Tversky Index Mean   : Ours = {df_res['Ours_Tversky'].mean():.4f} | Baseline = {df_res['Baseline_Tversky'].mean():.4f}")
print(f"Wasserstein Dist Mean: Ours = {df_res['Ours_Wasserstein'].mean():.4f} | Baseline = {df_res['Baseline_Wasserstein'].mean():.4f}")


## 7. Visual Inspection
We plot each processing step side-by-side to visually inspect the qualitative output.


In [ ]:
import matplotlib.pyplot as plt

for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    h, w = vis_data['visible'].shape
    
    print('\n' + '='*80)
    print(f'  FISSURE: {name}')
    print('='*80 + '\n')
    
    # Helper function to plot a single image with title and optional colorbar
    def plot_single(data, title, cmap='gray', is_rgba=False, scatter_pts=None, colorbar=False):
        plt.figure(figsize=(10, 6))
        if is_rgba:
            plt.imshow(data)
        else:
            im = plt.imshow(data, cmap=cmap)
            if colorbar:
                plt.colorbar(im)
        if scatter_pts is not None:
            plt.scatter(scatter_pts[:, 0], scatter_pts[:, 1], color='red', s=60, edgecolors='black', label='Sampled Prompts (N=12)')
            plt.legend()
        plt.title(title, fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    # 1. Visible Image
    plot_single(vis_data['visible'], f'{name} - 1. Visible Modality (Severe Gravel Clutter)', cmap='gray')
    
    # 2. Thermal Image
    plot_single(vis_data['infrared'], f'{name} - 2. Decoded Inverted Thermal', cmap='gray')
    
    # 3. Ground Truth Mask
    plot_single(vis_data['gt'], f'{name} - 3. Ground Truth (Fissure Target)', cmap='gray')
    
    # 4. Our Fused Multiscale Frangi
    plot_single(vis_data['ours_fused_frangi'], f'{name} - 4. Ours: Fused Frangi Response (Hessian Level)', cmap='magma', colorbar=True)
    
    # 5. Node Similarity Map
    plot_single(vis_data['ours_similarity'], f'{name} - 5. Ours: Graph Node Similarities (Max)', cmap='magma', colorbar=True)
    
    # 6. Graph Filtering Diagnostics
    rgba_tau = np.zeros((h, w, 4))
    rgba_tau[vis_data['ours_tau_mask'] > 0] = [1.0, 1.0, 1.0, 0.3] # White
    rgba_comp = np.zeros((h, w, 4))
    rgba_comp[vis_data['ours_comp_mask'] > 0] = [0.0, 0.5, 1.0, 0.8] # Blue
    
    plt.figure(figsize=(10, 6))
    plt.imshow(np.zeros((h, w)), cmap='gray')
    plt.imshow(rgba_tau)
    plt.imshow(rgba_comp)
    plt.title(f'{name} - 6. Ours: Pruned Nodes (White) & Active CCs (Blue)', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # 7. Betweenness Centrality
    plot_single(vis_data['ours_centrality'], f'{name} - 7. Ours: GPU Betweenness Centrality', cmap='hot', colorbar=True)
    
    # 8. Ours final output
    plot_single(vis_data['ours'], f'{name} - 8. Ours: Final Thickened Skeleton (5px)', cmap='hot')
    
    # 9. Baseline Standard Thermal Frangi
    plot_single(vis_data['baseline_frangi'], f'{name} - 9. Baseline: Standard Thermal Frangi Response', cmap='magma', colorbar=True)
    
    # 10. Top 0.5% Thresholded Mask
    plot_single(vis_data['baseline_binary'], f'{name} - 10. Baseline: Top 0.5% Thresholded Mask', cmap='gray')
    
    # 11. Skeleton with 12 Prompt Points
    plot_single(vis_data['baseline_skel'], f'{name} - 11. Baseline: Skeleton & Prompt Locations (Red)', cmap='gray', scatter_pts=vis_data['points'])
    
    # 12. Final Overlay comparison
    rgba_gt = np.zeros((h, w, 4))
    rgba_gt[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4] # White
    rgba_ours = np.zeros((h, w, 4))
    rgba_ours[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4] # Green
    rgba_base = np.zeros((h, w, 4))
    rgba_base[vis_data['baseline'] > 0] = [1.0, 0.0, 0.0, 0.4] # Red
    
    plt.figure(figsize=(10, 6))
    plt.imshow(vis_data['visible'], cmap='gray')
    plt.imshow(rgba_gt)
    plt.imshow(rgba_ours)
    plt.imshow(rgba_base)
    plt.title(f'{name} - 12. Overlay (White: GT, Green: Ours, Red: SAM 2)', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
